# 02 — Growing a tree, and reading it

*Notebook 2 of 5 — Decision Trees*

In the last notebook you found a tree's single best question. A whole tree is that one move, repeated.
Here you grow a small tree by hand, learn to read it as a flowchart, trace a penguin down to its leaf,
and watch it carve the feature space into boxes — then confirm your hand-grown tree is what the
library builds.

**Prerequisites**

- Notebook 01 — impurity and the best single split (the move we now repeat).
- Module 00 — the train/test split (NB 04), accuracy (NB 06), and cross-validation (NB 10).

**What you'll be able to do**

- Grow a small decision tree by recursively repeating the best-split rule.
- Read a fitted tree as a flowchart of yes/no questions.
- Trace a sample down to its leaf and read off the prediction.
- See the axis-aligned boxes a tree carves, and explain why growth is greedy.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier

from ml_course import datasets, viz
from ml_course.colors import CLASS_CYCLE, COLORS

np.random.seed(0)
viz.use_course_style()

# Binary penguin subset, raw millimetres; encode Gentoo = 1, Adelie = 0.
X, y = datasets.penguins_xy()
y_bin = (y == "Gentoo").astype(int).to_numpy()
flip = X["flipper_length_mm"].to_numpy()
bill = X["bill_length_mm"].to_numpy()
feats = {"flipper_length_mm": flip, "bill_length_mm": bill}


def gini(labels):
    """Gini impurity 1 - sum_k p_k^2 of a set of binary labels (0/1)."""
    labels = np.asarray(labels)
    if labels.size == 0:
        return 0.0
    p = labels.mean()
    return 1.0 - p**2 - (1.0 - p) ** 2


def best_split(mask):
    """Best (feature, threshold, impurity decrease) within the subset given by `mask`."""
    n = int(mask.sum())
    base = gini(y_bin[mask])
    best = (None, None, -1.0)
    for name, col in feats.items():
        values = np.sort(np.unique(col[mask]))
        for t in (values[:-1] + values[1:]) / 2.0:
            left = y_bin[mask & (col <= t)]
            right = y_bin[mask & (col > t)]
            if left.size == 0 or right.size == 0:
                continue
            dec = base - (left.size * gini(left) + right.size * gini(right)) / n
            if dec > best[2]:
                best = (name, float(t), dec)
    return best


print(f"{len(y_bin)} penguins; Adelie = {(y_bin == 0).sum()}, Gentoo = {y_bin.sum()}")

## Where we are

In Notebook 01 we measured the single best question on these penguins: **`flipper_length_mm ≤ 206`**,
which cut the root's Gini from 0.4948 down to 0.0216 — a 0.4732 drop. A decision tree adds only one new
idea: ask the same kind of question **again**, of each group the first split produced. Repeat that, and
a tree grows.

We stay on the binary penguin subset in raw millimetres (a tree is scale-invariant, as we saw), and we
grow only to **depth 2** — two questions deep. Knowing *when to stop* is the next notebook.

## One move, repeated

The root split sends each penguin left (`flipper ≤ 206`) or right (`flipper > 206`). Each side is now
its own group — its own node, with its own class mix — so we can ask it its own best question, exactly
the way we did for the root. Do that once on each child and we have a **depth-2** tree: the root, two
questions below it, and four groups at the bottom.

The groups at the very bottom, where we stop asking, are the **leaves**.

In [ ]:
root_feat, root_t, root_dec = best_split(np.ones(len(y_bin), dtype=bool))
print(f"root split : {root_feat} <= {root_t:.2f}  (Gini decrease {root_dec:.4f})")

left_mask = flip <= root_t
right_mask = flip > root_t
lf, lt, ld = best_split(left_mask)
rf, rt, rd = best_split(right_mask)
print(f"left child  (flipper <= {root_t:.0f}): split {lf} <= {lt:.2f}")
print(f"right child (flipper >  {root_t:.0f}): split {rf} <= {rt:.2f}")

leaves = {
    f"flipper<={root_t:.0f} & bill<={lt:.2f}": left_mask & (bill <= lt),
    f"flipper<={root_t:.0f} & bill>{lt:.2f}": left_mask & (bill > lt),
    f"flipper>{root_t:.0f} & bill<={rt:.2f}": right_mask & (bill <= rt),
    f"flipper>{root_t:.0f} & bill>{rt:.2f}": right_mask & (bill > rt),
}
print("\n4 leaves:")
for rule, m in leaves.items():
    sub = y_bin[m]
    pred = "Gentoo" if sub.mean() >= 0.5 else "Adelie"
    counts = f"{int((sub == 0).sum())} A / {int(sub.sum())} G"
    print(f"  {rule}: {counts}  ->  {pred}  (Gini {gini(sub):.4f})")

### Read the result

The recursion does something satisfying. The left child (`flipper ≤ 206`) is almost all Adélie — 149 of
them, with a single Gentoo — so its best question, `bill ≤ 47.20`, peels that one Gentoo off into its
own leaf. The right child is almost all Gentoo, and its question `bill ≤ 40.85` peels off one stray
Adélie.
We end with four leaves: three perfectly pure, and one (122 Gentoo with a single Adélie) almost pure.

## A leaf predicts its majority class

A finished tree is a **flowchart**. To classify a penguin you start at the top, answer each yes/no
question, and follow the matching branch down until you reach a leaf. The leaf's **majority class** is
the prediction. Let us draw the tree we grew.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.5))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")


def node(x, y, text, facecolor):
    ax.text(
        x,
        y,
        text,
        ha="center",
        va="center",
        fontsize=10,
        color=COLORS["text"],
        bbox=dict(boxstyle="round,pad=0.5", facecolor=facecolor, edgecolor=COLORS["text"]),
    )


def branch(parent, child, label):
    ax.annotate(
        "",
        xy=(child[0], child[1] + 0.07),
        xytext=(parent[0], parent[1] - 0.07),
        arrowprops=dict(arrowstyle="-|>", color=COLORS["text"], linewidth=1.2),
    )
    ax.text(
        (parent[0] + child[0]) / 2,
        (parent[1] + child[1]) / 2,
        label,
        ha="center",
        va="center",
        fontsize=9,
        color=COLORS["muted"],
    )


root, li, ri = (0.5, 0.86), (0.27, 0.52), (0.73, 0.52)
leaf_xy = [(0.10, 0.15), (0.40, 0.15), (0.60, 0.15), (0.90, 0.15)]
white, adelie, gentoo = COLORS["background"], CLASS_CYCLE[0], CLASS_CYCLE[1]

for parent, child, label in [
    (root, li, "yes"), (root, ri, "no"),
    (li, leaf_xy[0], "yes"), (li, leaf_xy[1], "no"),
    (ri, leaf_xy[2], "yes"), (ri, leaf_xy[3], "no"),
]:
    branch(parent, child, label)

node(*root, "flipper_length_mm\n≤ 206 mm?", white)
node(*li, "bill_length_mm\n≤ 47.20 mm?", white)
node(*ri, "bill_length_mm\n≤ 40.85 mm?", white)
node(*leaf_xy[0], "Adelie\n149 A / 0 G", adelie)
node(*leaf_xy[1], "Gentoo\n0 A / 1 G", gentoo)
node(*leaf_xy[2], "Adelie\n1 A / 0 G", adelie)
node(*leaf_xy[3], "Gentoo\n1 A / 122 G", gentoo)
plt.show()

### Read the figure

Read it top to bottom. The root asks about flipper length; "yes" (≤ 206) goes left, "no" goes right.
Each child asks one more question about bill length, and the four boxes at the bottom are the leaves,
coloured by the class they predict. Three leaves are a single colour — pure groups — and one is Gentoo
with a lone Adélie inside it.

In [ ]:
def trace(i):
    b, f = bill[i], flip[i]
    true = "Gentoo" if y_bin[i] == 1 else "Adelie"
    if f <= root_t:
        step2 = f"bill {b:.1f} {'<=' if b <= lt else '>'} {lt:.2f}"
        pred = "Adelie" if b <= lt else "Gentoo"
    else:
        step2 = f"bill {b:.1f} {'<=' if b <= rt else '>'} {rt:.2f}"
        pred = "Adelie" if b <= rt else "Gentoo"
    step1 = f"flipper {f:.0f} {'<=' if f <= root_t else '>'} {root_t:.0f}"
    flag = "" if pred == true else "   <- MISCLASSIFIED"
    print(f"penguin #{i}: bill {b:.1f}, flipper {f:.0f}  (true {true})")
    print(f"  {step1}  ->  {step2}  =>  predict {pred}{flag}")


clear_gentoo = int(np.flatnonzero((y_bin == 1) & (flip > 215))[0])
trace(clear_gentoo)
trace(128)

### Read the trace

The clear Gentoo answers "no" at the root (long flipper) and "no" again (long bill), landing in the
pure Gentoo leaf — an easy, confident call. The penguin at row 128 is an Adélie with an unusually long
flipper (210 mm, so "no" at the root) and a middling bill (> 40.85), so the tree sends it into the
Gentoo leaf and calls it wrong. It is the model's single training error, and it sits right where the
two species overlap — an honest mistake, not a broken rule.

## The tree carves the plane into boxes

Every question is a cut along one axis: `flipper ≤ 206` is a horizontal line, `bill ≤ 47.2` a vertical
one. Stack the cuts and the feature space is tiled into **axis-aligned rectangles** — one box per leaf.
Growing the tree deeper makes more, smaller boxes.

In [ ]:
# Fit on the raw arrays so the grid predictions inside plot_decision_boundary do not
# trip sklearn's feature-name warning; the DataFrame X still labels the axes.
d1 = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X.to_numpy(), y)
d2 = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X.to_numpy(), y)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
viz.plot_decision_boundary(d1, X, y, ax=ax1)
viz.plot_decision_boundary(d2, X, y, ax=ax2)
ax1.set_title("depth 1 — 2 boxes")
ax2.set_title("depth 2 — 4 boxes")
plt.show()

### Read the figure

On the left, the depth-1 tree makes a single horizontal cut at flipper = 206, splitting the plane into
two boxes. On the right, depth 2 cuts each of those halves again along bill length, giving four boxes
whose colours are the leaf predictions. More depth means more, smaller boxes — the dial we will turn in
Notebook 03. The one Adélie sitting inside the upper Gentoo box is that single misclassified
penguin.

In [ ]:
# By-hand depth-2 predictions (0/1), vectorized straight from the four leaves.
byhand = np.where(
    flip <= root_t,
    np.where(bill <= lt, 0, 1),
    np.where(bill <= rt, 0, 1),
)
sklearn_pred = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X, y_bin).predict(X)
print("by-hand == DecisionTreeClassifier(max_depth=2):", np.array_equal(byhand, sklearn_pred))
print(f"training accuracy: {(byhand == y_bin).mean():.4f}")

cv = StratifiedKFold(5, shuffle=True, random_state=0)
cv_depth2 = cross_val_score(DecisionTreeClassifier(max_depth=2, random_state=0), X, y_bin, cv=cv)
cv_full = cross_val_score(DecisionTreeClassifier(random_state=0), X, y_bin, cv=cv)
print(
    f"CV accuracy  depth-2: {cv_depth2.mean():.4f}   "
    f"|   unpruned full tree: {cv_full.mean():.4f}"
)

### Read the result

The by-hand predictions and `DecisionTreeClassifier(max_depth=2)` agree on every single penguin — the
library grows the very same tree by the very same greedy rule. The mechanism, not magic.

One number is worth pausing on. The depth-2 tree scores 0.9855 under cross-validation (module 00's
trick — re-score on several held-out folds and average; read it as "how well it does on data it did
not train on"), but the **unpruned** tree — grown until every leaf is pure — scores a touch *lower*,
0.9818. Growing more made it generalize slightly *worse* here. That is the first sign of overfitting,
and Notebook 03 makes it precise.

## Greedy, and what one more level buys

At every node we take the **locally** best split — the one with the biggest impurity drop right here.
We never look ahead. Finding the *globally* best tree is NP-hard (Hyafil & Rivest, 1976), so growing
greedily, one question at a time, is the standard and practical choice.

Does another level always help? Grow this tree to depth 3 and it gains a fifth leaf, but its training
accuracy stays at 0.9964 — the extra split separates points that were already both being called Gentoo.
Depth is a dial, and turning it up does not automatically buy accuracy. Knowing where to set it is the
next notebook.

## Your turn

1. **Read the flowchart.** A penguin has flipper 195 mm and bill 50 mm. Follow the questions from the
   top — which leaf does it reach, and what does the tree predict?
2. **Re-grow a child.** Call `best_split(right_mask)` and confirm the right child's best question is
   `bill ≤ 40.85`. Then say which leaf a penguin with flipper 215 mm and bill 39 mm lands in.
3. **One more level.** Fit `DecisionTreeClassifier(max_depth=3)`, count its leaves with
   `get_n_leaves()`, and explain why its training accuracy does not improve over the depth-2 tree.

## What you built

- A decision tree grown by **recursively** repeating Notebook 01's best-split move — root, two
  questions, four leaves.
- The rule that a **leaf predicts its majority class**, and the skill of **reading** a tree as a
  flowchart and **tracing** a sample to its leaf.
- The **axis-aligned boxes** a tree carves, and how depth multiplies them.
- A parity check: your hand-grown tree is exactly `DecisionTreeClassifier(max_depth=2)` — and a first
  hint that an unpruned tree can generalize worse.

**Vocabulary:** recursion · greedy growth · internal node · branch · leaf · majority-class prediction ·
flowchart · axis-aligned box · parity.

## Going further (optional)

- **`plot_tree`.** scikit-learn ships `sklearn.tree.plot_tree`, which draws any fitted tree (it shades
  nodes by class and purity in its own palette). We drew ours by hand to keep the class colours
  matching the rest of the course.
- **Binary, top-down.** The CART trees we use always split a node in two and grow from the root down;
  the greedy choice is what makes that fast.

## References

- Breiman, Friedman, Olshen & Stone (1984). *Classification and Regression Trees (CART).* Wadsworth.
- Hyafil, L. & Rivest, R. L. (1976). *Constructing optimal binary decision trees is NP-complete.*
  Information Processing Letters 5(1), 15–17. DOI: 10.1016/0020-0190(76)90095-8.
- Hastie, Tibshirani & Friedman (2009). *The Elements of Statistical Learning*, §9.2.
  DOI: 10.1007/978-0-387-84858-7.
- James, Witten, Hastie & Tibshirani (2021). *An Introduction to Statistical Learning*, §8.1.
  DOI: 10.1007/978-1-0716-1418-1.

*Previous: 01 — A question that splits the data: impurity · Next: 03 — Overfitting & pruning: depth is
the complexity dial.*